# Four GraphRAG retrieval patterns

This notebook asks what graph enrichment adds to vector and keyword retrieval. Phases 1 and 2 configure the environment and build a small, deterministic graph from NVIDIA and Amazon 10-K filings.

## 1. Configure

Load local settings, keep the embedding model and dimensions in one place, validate the two bundled filings, and connect to Neo4j. Use a dedicated empty database for this demo.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import BedrockEmbeddings
from pypdf import PdfReader

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").is_dir():
    PROJECT_DIR = PROJECT_DIR / "sec-filings-graphrag-demo"
if not (PROJECT_DIR / "data").is_dir():
    raise FileNotFoundError("Run this notebook from its directory or the repository root.")

load_dotenv(PROJECT_DIR / ".env")

EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"
EMBEDDING_DIMENSIONS = 1024
VECTOR_INDEX_NAME = "chunkEmbeddings"
FULLTEXT_INDEX_NAME = "search_chunks"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
CHUNK_SIZE = 2500
CHUNK_OVERLAP = 250
RESET_DATABASE = False

FILINGS = [
    {
        "path": "data/0001045810-23-000017.pdf",
        "company": "NVIDIA CORPORATION",
        "ticker": "NVDA",
        "cik": "1045810",
    },
    {
        "path": "data/0001018724-23-000004.pdf",
        "company": "AMAZON",
        "ticker": "AMZN",
        "cik": "1018724",
    },
]

required_settings = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError(
        f"Missing {', '.join(missing_settings)}. Copy .env.sample to .env and fill it in."
    )

placeholder_settings = [
    name
    for name in required_settings
    if "replace-me" in os.environ[name] or "your-database-id" in os.environ[name]
]
if placeholder_settings:
    raise RuntimeError(
        f"Replace the sample values for {', '.join(placeholder_settings)} in .env."
    )

required_filing_fields = {"path", "company", "ticker", "cik"}
if len(FILINGS) != 2:
    raise ValueError("This demo requires exactly two filing metadata records.")
if len({filing["cik"] for filing in FILINGS}) != len(FILINGS):
    raise ValueError("Each filing must have a distinct CIK.")
for filing in FILINGS:
    missing_fields = required_filing_fields - filing.keys()
    if missing_fields or not all(filing.get(field) for field in required_filing_fields):
        raise ValueError(f"Invalid filing metadata: {filing!r}")
    filing_path = PROJECT_DIR / filing["path"]
    if not filing_path.is_file():
        raise FileNotFoundError(f"Missing bundled filing: {filing_path}")

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)
driver.verify_connectivity()
embedder = BedrockEmbeddings(
    model_id=EMBEDDING_MODEL_ID,
    dimensions=EMBEDDING_DIMENSIONS,
    region_name=AWS_REGION,
)

print(f"Connected to Neo4j; found {len(FILINGS)} filings.")
print(f"Embedder: {EMBEDDING_MODEL_ID} ({EMBEDDING_DIMENSIONS} dimensions)")

## 2. Ingest

The default is deliberately safe: if Neo4j contains any nodes, this cell stops before writing. Setting `RESET_DATABASE = True` in the configuration cell deletes every node and relationship and drops the `chunkEmbeddings` and `search_chunks` demo indexes in the selected database.

Both PDFs are fully extracted, chunked, and embedded before the graph write begins. A failure in either filing therefore stops the run without creating a partial two-filing graph.

In [ ]:
def extract_pdf_text(path: Path) -> str:
    reader = PdfReader(path)
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()
        if text and text.strip():
            pages.append(text.strip())
        else:
            print(f"Warning: {path.name} page {page_number} contained no extractable text.")
    document_text = "\n\n".join(pages)
    if not document_text:
        raise ValueError(f"No text could be extracted from {path.name}.")
    return document_text


def split_text(
    text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP
) -> list[str]:
    if chunk_size <= overlap:
        raise ValueError("CHUNK_SIZE must be larger than CHUNK_OVERLAP.")

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = text.rfind(" ", start + int(chunk_size * 0.8), end)
            if boundary > start:
                end = boundary
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = end - overlap
    return chunks


node_count = driver.execute_query(
    "MATCH (n) RETURN count(n) AS count",
    result_transformer_=lambda result: result.single()["count"],
)
if node_count and not RESET_DATABASE:
    raise RuntimeError(
        f"The selected database contains {node_count} nodes. Use a dedicated empty "
        "database or explicitly set RESET_DATABASE = True."
    )

if RESET_DATABASE:
    driver.execute_query(f"DROP INDEX {VECTOR_INDEX_NAME} IF EXISTS")
    driver.execute_query(f"DROP INDEX {FULLTEXT_INDEX_NAME} IF EXISTS")
    driver.execute_query("MATCH (n) DETACH DELETE n")
    print("Reset complete: all nodes, relationships, and demo indexes were deleted.")

rows = []
for filing in FILINGS:
    path = PROJECT_DIR / filing["path"]
    try:
        chunks = split_text(extract_pdf_text(path))
        if not chunks:
            raise ValueError("Text splitting produced no chunks.")

        embeddings = [embedder.embed_query(chunk) for chunk in chunks]
        if any(len(embedding) != EMBEDDING_DIMENSIONS for embedding in embeddings):
            raise ValueError(
                f"Expected {EMBEDDING_DIMENSIONS}-dimensional embeddings from "
                f"{EMBEDDING_MODEL_ID}."
            )

        document_uid = path.stem
        for position, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
            uid = f"{filing['cik']}-{position:04d}"
            next_uid = (
                f"{filing['cik']}-{position + 1:04d}"
                if position + 1 < len(chunks)
                else None
            )
            rows.append(
                {
                    "uid": uid,
                    "next_uid": next_uid,
                    "position": position,
                    "text": chunk,
                    "embedding": embedding,
                    "document_uid": document_uid,
                    "document_name": path.name,
                    "company": filing["company"],
                    "ticker": filing["ticker"],
                    "cik": filing["cik"],
                }
            )
        print(f"Prepared {len(chunks)} chunks for {filing['ticker']}.")
    except Exception as exc:
        raise RuntimeError(f"Failed to prepare {filing['ticker']} filing {path.name}.") from exc


def write_graph(transaction, graph_rows):
    transaction.run(
        """
        UNWIND $rows AS row
        MERGE (company:Company {cik: row.cik})
        SET company.name = row.company, company.ticker = row.ticker
        MERGE (document:Document {uid: row.document_uid})
        SET document.name = row.document_name, document.cik = row.cik
        MERGE (company)-[:FILED]->(document)
        MERGE (chunk:Chunk {uid: row.uid})
        SET chunk.position = row.position,
            chunk.text = row.text,
            chunk.embedding = row.embedding
        MERGE (chunk)-[:FROM_DOCUMENT]->(document)
        """,
        rows=graph_rows,
    ).consume()
    transaction.run(
        """
        UNWIND $rows AS row
        WITH row WHERE row.next_uid IS NOT NULL
        MATCH (chunk:Chunk {uid: row.uid})
        MATCH (following:Chunk {uid: row.next_uid})
        MERGE (chunk)-[:NEXT_CHUNK]->(following)
        """,
        rows=graph_rows,
    ).consume()


with driver.session() as session:
    session.execute_write(write_graph, rows)

print(f"Ingestion complete: wrote {len(rows)} chunks from {len(FILINGS)} filings.")

## 3. Index and verify

Create the vector and full-text indexes, wait until both are online, then check the deterministic graph shape and required chunk properties before retrieval begins.

In [ ]:
driver.execute_query(
    f"""
    CREATE VECTOR INDEX {VECTOR_INDEX_NAME} IF NOT EXISTS
    FOR (chunk:Chunk) ON (chunk.embedding)
    OPTIONS {{indexConfig: {{
        `vector.dimensions`: {EMBEDDING_DIMENSIONS},
        `vector.similarity_function`: 'cosine'
    }}}}
    """
)
driver.execute_query(
    f"""
    CREATE FULLTEXT INDEX {FULLTEXT_INDEX_NAME} IF NOT EXISTS
    FOR (chunk:Chunk) ON EACH [chunk.text]
    """
)
driver.execute_query("CALL db.awaitIndexes(300)")

index_states = driver.execute_query(
    """
    SHOW INDEXES YIELD name, type, state
    WHERE name IN $index_names
    RETURN name, type, state
    ORDER BY name
    """,
    index_names=[VECTOR_INDEX_NAME, FULLTEXT_INDEX_NAME],
    result_transformer_=lambda result: [dict(record) for record in result],
)
assert len(index_states) == 2, index_states
assert all(index["state"] == "ONLINE" for index in index_states), index_states

counts = driver.execute_query(
    """
    CALL { MATCH (n:Company) RETURN count(n) AS companies }
    CALL { MATCH (n:Document) RETURN count(n) AS documents }
    CALL { MATCH (n:Chunk) RETURN count(n) AS chunks }
    CALL { MATCH ()-[r:FILED]->() RETURN count(r) AS filed }
    CALL { MATCH ()-[r:FROM_DOCUMENT]->() RETURN count(r) AS from_document }
    CALL { MATCH ()-[r:NEXT_CHUNK]->() RETURN count(r) AS next_chunk }
    RETURN companies, documents, chunks, filed, from_document, next_chunk
    """,
    result_transformer_=lambda result: dict(result.single()),
)
invalid_chunks = driver.execute_query(
    """
    MATCH (chunk:Chunk)
    OPTIONAL MATCH (chunk)-[:FROM_DOCUMENT]->(document:Document)
    WITH chunk, count(document) AS document_links
    WHERE chunk.uid IS NULL
       OR chunk.text IS NULL
       OR trim(chunk.text) = ''
       OR chunk.position IS NULL
       OR chunk.embedding IS NULL
       OR size(chunk.embedding) <> $dimensions
       OR document_links <> 1
    RETURN count(chunk) AS count
    """,
    dimensions=EMBEDDING_DIMENSIONS,
    result_transformer_=lambda result: result.single()["count"],
)
invalid_sequence_links = driver.execute_query(
    """
    MATCH (chunk:Chunk)-[:NEXT_CHUNK]->(following:Chunk)
    MATCH (chunk)-[:FROM_DOCUMENT]->(document:Document)
    MATCH (following)-[:FROM_DOCUMENT]->(following_document:Document)
    WHERE document <> following_document
       OR following.position <> chunk.position + 1
    RETURN count(*) AS count
    """,
    result_transformer_=lambda result: result.single()["count"],
)

assert counts["companies"] == 2, counts
assert counts["documents"] == 2, counts
assert counts["chunks"] == len(rows), counts
assert counts["filed"] == counts["documents"], counts
assert counts["from_document"] == counts["chunks"], counts
assert counts["next_chunk"] == counts["chunks"] - counts["documents"], counts
assert invalid_chunks == 0, f"Found {invalid_chunks} invalid chunks."
assert invalid_sequence_links == 0, (
    f"Found {invalid_sequence_links} invalid NEXT_CHUNK relationships."
)

print("Indexes are online and graph integrity checks passed.")
for index in index_states:
    print(f"  {index['name']}: {index['state']} ({index['type']})")
for name, count in counts.items():
    print(f"  {name}: {count}")

## 4. Define retrieval

Define four strategies behind one `retrieve` helper. Vector and keyword results expose only the shared result fields. The two graph-enriched strategies also traverse to the filing and neighboring chunks, adding source and local document context without changing the underlying vector ranking.

In [ ]:
from neo4j_graphrag.retrievers import (
    HybridCypherRetriever,
    VectorCypherRetriever,
    VectorRetriever,
)
from neo4j_graphrag.types import RetrieverResult, RetrieverResultItem


def vector_result_formatter(record):
    node = record["node"]
    return RetrieverResultItem(
        content=node["text"],
        metadata={"score": record["score"], "chunk_id": node["uid"]},
    )


def graph_result_formatter(record):
    return RetrieverResultItem(
        content=record["text"],
        metadata={"score": record["score"], **record["metadata"]},
    )


GRAPH_ENRICHMENT_QUERY = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)<-[:FILED]-(company:Company)
OPTIONAL MATCH (previous:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(following:Chunk)
RETURN node.text AS text,
       score,
       {
         chunk_id: node.uid,
         document: doc.name,
         company: company.name,
         ticker: company.ticker,
         previous_text: previous.text,
         next_text: following.text
       } AS metadata
"""

vector_retriever = VectorRetriever(
    driver,
    index_name=VECTOR_INDEX_NAME,
    embedder=embedder,
    return_properties=["uid", "text"],
    result_formatter=vector_result_formatter,
)
vector_graph_retriever = VectorCypherRetriever(
    driver,
    index_name=VECTOR_INDEX_NAME,
    retrieval_query=GRAPH_ENRICHMENT_QUERY,
    embedder=embedder,
    result_formatter=graph_result_formatter,
)
hybrid_graph_retriever = HybridCypherRetriever(
    driver,
    vector_index_name=VECTOR_INDEX_NAME,
    fulltext_index_name=FULLTEXT_INDEX_NAME,
    retrieval_query=GRAPH_ENRICHMENT_QUERY,
    embedder=embedder,
    result_formatter=graph_result_formatter,
)

KEYWORD_QUERY = """
CALL db.index.fulltext.queryNodes($index_name, $query, {limit: $top_k})
YIELD node, score
WHERE node.uid IS NOT NULL
RETURN node.text AS text, score, {chunk_id: node.uid} AS metadata
ORDER BY score DESC, node.uid
LIMIT $top_k
"""


def keyword_retrieve(question, top_k):
    records = driver.execute_query(
        KEYWORD_QUERY,
        index_name=FULLTEXT_INDEX_NAME,
        query=question,
        top_k=top_k,
        result_transformer_=lambda result: list(result),
    )
    return RetrieverResult(
        items=[graph_result_formatter(record) for record in records],
        metadata={"__retriever": "FulltextQuery"},
    )


RETRIEVAL_STRATEGIES = {
    "Vector": lambda question, top_k: vector_retriever.search(
        query_text=question, top_k=top_k
    ),
    "Vector + graph": lambda question, top_k: vector_graph_retriever.search(
        query_text=question, top_k=top_k
    ),
    "Keyword": keyword_retrieve,
    "Vector + keyword + graph": lambda question, top_k: hybrid_graph_retriever.search(
        query_text=question, top_k=top_k
    ),
}


def retrieve(pattern, question, top_k=3, preview_length=160):
    if pattern not in RETRIEVAL_STRATEGIES:
        choices = ", ".join(RETRIEVAL_STRATEGIES)
        raise ValueError(f"Unknown retrieval pattern {pattern!r}; choose from {choices}.")

    result = RETRIEVAL_STRATEGIES[pattern](question, top_k)
    normalized = []
    for rank, item in enumerate(result.items, start=1):
        metadata = item.metadata or {}
        row = {
            "rank": rank,
            "score": metadata.get("score"),
            "chunk_id": metadata.get("chunk_id"),
            "preview": " ".join(str(item.content).split())[:preview_length],
        }
        for field in (
            "document",
            "company",
            "ticker",
            "previous_text",
            "next_text",
        ):
            if field in metadata:
                row[field] = metadata[field]
        normalized.append(row)
    return normalized


print("Ready: " + ", ".join(RETRIEVAL_STRATEGIES))

## Next

Phase 4 runs both questions through `retrieve`, displays compact comparison tables, and generates one grounded answer with the hybrid graph retriever.